# CTAO sky-background API example

Use the same NSB public API at two representative CTAO locations. These approximate
coordinates and planning profiles illustrate geographic configuration; they do not
supply an instrument calibration. Radiance is in photons cm⁻² ns⁻¹ sr⁻¹.

Build/install NSB from this checkout (`python -m pip install .`), then install
`astropy numpy matplotlib ipykernel` in the same environment. Restart the kernel
after changing NSB builds. This notebook needs no datasets or internal helper package.

**v0.2.0:** Automatic resolves to PALACE Oxygen, with **Experimental** maturity,
standard continuum geometry, and **zenith distance ≤60°**. The Paranal-derived
model is a spatial extrapolation at other sites. Invalid Oxygen geometry raises
an error; no continuum fallback occurs. Explicit continuum preserves historical behavior.

For observations, bootstrap intervals, provenance receipts and scientific admission,
see [nsb-validation PR #23](https://github.com/VPRamon/nsb-validation/pull/23).
Patat informed development; LVM absolute equivalence remains pending. Abastumani
has limitations for absolute admission. These examples are not calibrated camera rates.

Code: SPDX-License-Identifier: AGPL-3.0-only. Copyright (C) 2026 Vallés Puig, Ramon.


In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Copyright (C) 2026 Vallés Puig, Ramon
from datetime import datetime, timezone
import astropy.units as u
from astropy.coordinates import AltAz, EarthLocation, SkyCoord
from astropy.time import Time
from astropy.utils import iers
import matplotlib.pyplot as plt
import numpy as np
import nsb
import siderust

iers.conf.auto_download = False
instant = datetime(2024, 1, 15, 2, tzinfo=timezone.utc)
sites = {
    "CTAO South": (-70.404, -24.627, 2150, nsb.SiteProfile.CTA_SOUTH),
    "CTAO North": (-17.890, 28.762, 2200, nsb.SiteProfile.CTA_NORTH),
}

def direction_at(location, zd):
    # Transform actual local altitude to ICRS, without refraction (pressure=0).
    sky = SkyCoord(az=0*u.deg, alt=(90-zd)*u.deg,
                   frame=AltAz(obstime=Time(instant), location=location, pressure=0*u.hPa)).icrs
    return siderust.Direction(ra_deg=float(sky.ra.deg), dec_deg=float(sky.dec.deg))


## Automatic, Oxygen and continuum

Hold solar flux and site configuration fixed. The default and explicit Oxygen
must agree exactly; metadata preserves the requested selection. Fixing F10.7
here is a reproducible input choice, not a site calibration.


In [ ]:
evaluators, geometry = {}, {}
for name, (lon, lat, height, profile) in sites.items():
    observer = siderust.Observer(lon_deg=lon, lat_deg=lat, height_m=height)
    location = EarthLocation.from_geodetic(lon*u.deg, lat*u.deg, height*u.m)
    geometry[name] = (observer, location)
    config = (nsb.NsbModelConfig.generic_clear_sky()
              .with_site_profile(profile).with_solar_radio_flux_sfu(130))
    models = {
        "Automatic": nsb.NsbEvaluator(config),
        "Oxygen": nsb.NsbEvaluator(config.with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_OXYGEN_V1)),
        "Continuum": nsb.NsbEvaluator(config.with_airglow_model(nsb.AirglowModel.PARANAL_PALACE_V1)),
    }
    evaluators[name] = models
    query = nsb.PointQuery(observer, instant, direction_at(location, 0), components=nsb.ComponentMask.AIRGLOW)
    values = {key: evaluator.integrate_band(query, 300, 650) for key, evaluator in models.items()}
    assert values["Automatic"] == values["Oxygen"]
    assert values["Oxygen"] > values["Continuum"]
    for key in ("Automatic", "Oxygen"):
        metadata = models[key].evaluate(query).components[0].metadata
        print(name, key, metadata.airglow_selection_kind, metadata.airglow_resolved_model)
    print(name, values)


## Components and zenith distance

The left panel separates public sky radiance components at local zenith. The right
panel compares Airglow models within the Oxygen domain. The Moon contribution
reflects the actual geometry at this instant. These are illustrative conditions,
not an observing-window recommendation or detector simulation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
zenith_distances = np.linspace(0, 59, 16)  # Leave margin for frame transformations at 60°.
for index, (name, models) in enumerate(evaluators.items()):
    observer, location = geometry[name]
    query = nsb.PointQuery(observer, instant, direction_at(location, 0))
    components = models["Automatic"].evaluate(query).components
    axes[0].bar(np.arange(len(components)) + index*0.38,
                [c.integrated_photons_cm2_ns_sr for c in components], width=0.38, label=name)
    axes[0].set_xticks(np.arange(len(components))+0.19, [c.name for c in components])
    for label in ("Automatic", "Continuum"):
        values = [models[label].integrate_band(
            nsb.PointQuery(observer, instant, direction_at(location, float(zd)),
                           components=nsb.ComponentMask.AIRGLOW), 300, 650)
                  for zd in zenith_distances]
        axes[1].plot(zenith_distances, values, label=f"{name}: {label}")
axes[0].set_title("Sky components at zenith (300–650 nm)")
axes[1].set_title("Airglow (300–650 nm)")
axes[1].set_xlabel("Zenith distance (degrees)")
for ax in axes:
    ax.set_ylabel("photons cm⁻² ns⁻¹ sr⁻¹")
    ax.legend(fontsize=8)
plt.show()


## Explicit domain errors

Planning tracks and sky scans can leave the Oxygen domain. Restrict such queries
to ZD ≤60°, or deliberately select continuum for broader coverage. The latter
changes the physical model and must be recorded in any comparison.


In [ ]:
for name, models in evaluators.items():
    observer, location = geometry[name]
    query = nsb.PointQuery(observer, instant, direction_at(location, 70), components=nsb.ComponentMask.AIRGLOW)
    for label in ("Automatic", "Oxygen"):
        try:
            models[label].integrate_band(query, 300, 650)
        except nsb.OutOfRangeError as error:
            print(name, label, str(error))
        else:
            raise AssertionError("Out-of-domain Oxygen unexpectedly succeeded")
    assert np.isfinite(models["Continuum"].integrate_band(query, 300, 650))
